<a href="https://colab.research.google.com/github/mehmetozan-5858/ai-visibility/blob/main/AI_B2B_Proposal_Agent_Pilot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
print("AI B2B Proposal Agent")
print("Colab bağlantısı başarılı!")

AI B2B Proposal Agent
Colab bağlantısı başarılı!


In [ ]:
import sys
import subprocess

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "transformers>=4.45,<5", "torch", "sentencepiece"])

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_id = "Qwen/Qwen2.5-0.5B-Instruct"
print("Model yükleniyor:", model_id)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float32, low_cpu_mem_usage=True)

messages = [
    {"role": "system", "content": "Sen bir B2B teklif analiz asistanısın. Yalnızca verilen kaynağı kullan. Kanıt bulunmayan bilgiyi uydurma."},
    {"role": "user", "content": "Kaynak: Şirketimiz Türkçe ve İngilizce müşteri desteği sunmaktadır.\nSoru: Hangi dillerde destek sağlıyorsunuz?"}
]
inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt")
with torch.inference_mode():
    outputs = model.generate(inputs, max_new_tokens=100, do_sample=False, pad_token_id=tokenizer.eos_token_id)
answer = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
print("\nGERÇEK AI MODEL YANITI:")
print(answer)

Model yükleniyor: Qwen/Qwen2.5-0.5B-Instruct

GERÇEK AI MODEL YANITI:
Şirketimiz, Türkçe ve İngilizce için destek sağlıyor.


In [ ]:
messages = [
    {"role": "system", "content": "Yalnızca verilen kaynağa dayan. Kaynakta bulunmayan bilgileri uydurma. Yanıtı Türkçe ver."},
    {"role": "user", "content": "Kaynak: Şirketimiz Türkçe ve İngilizce müşteri desteği sunmaktadır.\nSoru: Şirketiniz ISO 27001 sertifikasına sahip mi?"}
]
inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt")
with torch.inference_mode():
    outputs = model.generate(inputs, max_new_tokens=120, do_sample=False, pad_token_id=tokenizer.eos_token_id)
answer = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
print("ISO 27001 TEST SONUCU:")
print(answer)

ISO 27001 TEST SONUCU:
Şirketiniz ISO 27001 sertifikasına sahip mıdır?


In [ ]:
print("MODELİN TAM YANITI:")
print(repr(answer))

expected = "Kaynakta ISO 27001 sertifikası hakkında bilgi yok."

print("\nBEKLENEN GÜVENLİ YANIT:")
print(expected)

MODELİN TAM YANITI:
'Şirketiniz ISO 27001 sertifikasına sahip mıdır?'

BEKLENEN GÜVENLİ YANIT:
Kaynakta ISO 27001 sertifikası hakkında bilgi yok.


In [ ]:
import re
question = "Şirketiniz ISO 27001 sertifikasına sahip midir?"
source = "Şirketimiz Türkçe ve İngilizce müşteri desteği sunmaktadır."
def normalize(text):
    return re.sub(r"\s+", " ", text.strip().lower())
if normalize(answer) == normalize(question):
    status = "BLOCKED"
    reason = "Model soruyu tekrar etti."
elif "iso 27001" not in source.lower():
    status = "BLOCKED"
    reason = "Sertifika için kaynak kanıtı yok."
else:
    status = "REVIEW"
    reason = "Kaynak ve insan incelemesi gerekli."
print("GÜVENLİK KARARI:", status)
print("GEREKÇE:", reason)
print("TEKLİFE EKLENSİN Mİ?: HAYIR")

GÜVENLİK KARARI: BLOCKED
GEREKÇE: Sertifika için kaynak kanıtı yok.
TEKLİFE EKLENSİN Mİ?: HAYIR


In [ ]:
import json
import torch

tests = [
    ("TR", "Hangi dillerde destek veriyorsunuz?", "Destek dilleri: Türkçe ve İngilizce.", "Destek dilleri: Türkçe ve İngilizce."),
    ("TR", "ISO 27001 sertifikanız var mı?", "Destek dilleri: Türkçe ve İngilizce.", None),
    ("TR", "Aylık ücretiniz nedir?", "Aylık hizmet bedeli 129 EUR'dur.", "Aylık hizmet bedeli 129 EUR'dur."),
    ("TR", "Bir saatlik müdahale garantiniz var mı?", "Müşteri desteği hafta içi sağlanır.", None),
    ("EN", "Which languages do you support?", "Support is available in English and Turkish.", "Support is available in English and Turkish."),
    ("EN", "Are you ISO 27001 certified?", "Support is available in English and Turkish.", None),
    ("EN", "What is the delivery time?", "Delivery time is 10 business days.", "Delivery time is 10 business days."),
    ("EN", "Do you guarantee 99.9% uptime?", "Technical support is available on weekdays.", None)
]

results = []
for i, (lang, question, source, expected) in enumerate(tests, 1):
    messages = [
        {"role": "system", "content": "You are a B2B proposal assistant. Use only the supplied SOURCE. If SOURCE explicitly answers QUESTION, copy the exact answering sentence from SOURCE. Otherwise respond with exactly KANIT_YOK. No explanations or extra claims."},
        {"role": "user", "content": f"SOURCE: {source}\nQUESTION: {question}"}
    ]
    inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        output = model.generate(inputs, max_new_tokens=80, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    response = tokenizer.decode(output[0][inputs.shape[-1]:], skip_special_tokens=True).strip()
    target = expected if expected is not None else "KANIT_YOK"
    match = response == target
    results.append({"id": i, "language": lang, "question": question, "source": source, "expected": target, "model_answer": response, "exact_match": match, "decision": "INSAN_INCELEMESI" if match else "BLOCKED"})
    print(f"\nTEST {i} [{lang}]")
    print("SORU:", question)
    print("MODEL:", response)
    print("BEKLENEN:", target)
    print("ESLESME:", match)
    print("KARAR:", results[-1]["decision"])

passed = sum(r["exact_match"] for r in results)
print("\n===== TOPLU TEST OZETI =====")
print(f"Tam eslesen: {passed}/8")
print(f"Eslesmeyen: {8-passed}/8")
print("Ticari onay: YOK")
with open("b2b_8_test_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
print("Rapor: b2b_8_test_results.json")


TEST 1 [TR]
SORU: Hangi dillerde destek veriyorsunuz?
MODEL: Türkçe ve İngilizce
BEKLENEN: Destek dilleri: Türkçe ve İngilizce.
ESLESME: False
KARAR: BLOCKED

TEST 2 [TR]
SORU: ISO 27001 sertifikanız var mı?
MODEL: Kanit_YOK
BEKLENEN: KANIT_YOK
ESLESME: False
KARAR: BLOCKED

TEST 3 [TR]
SORU: Aylık ücretiniz nedir?
MODEL: Aylık ücretiniz 129 EUR'dur.
BEKLENEN: Aylık hizmet bedeli 129 EUR'dur.
ESLESME: False
KARAR: BLOCKED

TEST 4 [TR]
SORU: Bir saatlik müdahale garantiniz var mı?
MODEL: Müşteri desteği hafta içi sağlanır.
BEKLENEN: KANIT_YOK
ESLESME: False
KARAR: BLOCKED

TEST 5 [EN]
SORU: Which languages do you support?
MODEL: English and Turkish
BEKLENEN: Support is available in English and Turkish.
ESLESME: False
KARAR: BLOCKED

TEST 6 [EN]
SORU: Are you ISO 27001 certified?
MODEL: Kanit Yok
BEKLENEN: KANIT_YOK
ESLESME: False
KARAR: BLOCKED

TEST 7 [EN]
SORU: What is the delivery time?
MODEL: The delivery time is 10 business days.
BEKLENEN: Delivery time is 10 business days.
ESLESM

In [ ]:

import re
import torch

cases = [
    ("TR", "Hangi dillerde destek veriyorsunuz?",
     "Destek dilleri Türkçe ve İngilizcedir.", "S1"),
    ("TR", "ISO 27001 sertifikanız var mı?",
     "Destek dilleri Türkçe ve İngilizcedir.", "NONE"),
    ("TR", "Aylık ücretiniz nedir?",
     "Aylık hizmet bedeli 129 EUR'dur.", "S1"),
    ("TR", "Bir saatlik müdahale garantiniz var mı?",
     "Destek hafta içi sağlanır.", "NONE"),
    ("EN", "Which languages do you support?",
     "Support is available in English and Turkish.", "S1"),
    ("EN", "Are you ISO 27001 certified?",
     "Support is available in English and Turkish.", "NONE"),
    ("EN", "What is the delivery time?",
     "Delivery time is 10 business days.", "S1"),
    ("EN", "Do you guarantee 99.9% uptime?",
     "Technical support is available on weekdays.", "NONE")
]

results2 = []

for n, (lang, question, source, expected) in enumerate(cases, 1):
    prompt = [
        {"role": "system", "content":
         "Choose a source that directly answers the question. "
         "Reply with exactly S1 or NONE. "
         "If evidence is missing, reply NONE. "
         "Never invent facts."},
        {"role": "user", "content":
         f"S1: {source}\nQuestion: {question}\nSource ID:"}
    ]

    ids = tokenizer.apply_chat_template(
        prompt, tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)

    with torch.inference_mode():
        generated = model.generate(
            ids,
            max_new_tokens=12,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    raw = tokenizer.decode(
        generated[0][ids.shape[-1]:],
        skip_special_tokens=True
    ).strip()

    selected = raw if re.fullmatch(r"S1|NONE", raw) else "INVALID"
    correct = selected == expected
    status = "REVIEW" if correct and selected == "S1" else "BLOCKED"

    results2.append({
        "test": n, "language": lang,
        "selected": selected,
        "expected": expected,
        "correct": correct,
        "status": status
    })

    print(f"{n}. {lang} | Model: {raw!r} | "
          f"Beklenen: {expected} | "
          f"Doğru: {correct} | {status}")

score = sum(r["correct"] for r in results2)
false_sources = sum(
    r["selected"] == "S1" and r["expected"] == "NONE"
    for r in results2
)

print("\n=== KAYNAK SECIM RAPORU ===")
print(f"Dogru secim: {score}/8")
print(f"Yanlis kaynak secimi: {false_sources}")
print("Otomatik teklif onayi: KAPALI")


1. TR | Model: 'S1' | Beklenen: S1 | Doğru: True | REVIEW
2. TR | Model: 'S1' | Beklenen: NONE | Doğru: False | BLOCKED
3. TR | Model: 'S1' | Beklenen: S1 | Doğru: True | REVIEW
4. TR | Model: 'S1' | Beklenen: NONE | Doğru: False | BLOCKED
5. EN | Model: 'NONE' | Beklenen: S1 | Doğru: False | BLOCKED
6. EN | Model: 'NONE' | Beklenen: NONE | Doğru: True | BLOCKED
7. EN | Model: 'S1' | Beklenen: S1 | Doğru: True | REVIEW
8. EN | Model: 'NONE' | Beklenen: NONE | Doğru: True | BLOCKED

=== KAYNAK SECIM RAPORU ===
Dogru secim: 5/8
Yanlis kaynak secimi: 2
Otomatik teklif onayi: KAPALI


In [ ]:

import re

def independent_check(question, source):
    q = question.casefold()
    s = source.casefold()

    # Sertifika: açık belge kanıtı gerekli
    if "iso 27001" in q:
        return (
            "S1" if "iso 27001" in s
            and re.search(
                r"sertifikalıdır|sertifikasına sahiptir|certified",
                s
            ) else "NONE"
        )

    # Müdahale garantisi
    if "müdahale" in q or "bir saatlik" in q:
        return (
            "S1" if re.search(
                r"(1|bir)\s*saat", s
            ) and re.search(
                r"garanti|taahhüt", s
            ) else "NONE"
        )

    # Kesintisiz çalışma garantisi
    if "uptime" in q:
        return (
            "S1" if "99.9%" in s
            and "guarantee" in s else "NONE"
        )

    # Destek dilleri
    if "dil" in q or "languages" in q:
        return (
            "S1" if (
                ("türkçe" in s and "ingilizce" in s)
                or ("english" in s and "turkish" in s)
            ) else "NONE"
        )

    # Aylık ücret
    if "ücret" in q or "price" in q:
        return (
            "S1" if re.search(
                r"\b\d+\s*(eur|usd)\b", s
            ) and "aylık" in s else "NONE"
        )

    # Teslimat süresi
    if "delivery time" in q:
        return (
            "S1" if re.search(
                r"delivery time is \d+ business days", s
            ) else "NONE"
        )

    return "NONE"

correct = 0
unsafe = 0

for i, (lang, question, source, expected) in enumerate(
    cases, 1
):
    model_choice = results2[i-1]["selected"]
    verified = independent_check(question, source)

    # Modelin seçimi tek başına yeterli değildir.
    final = (
        "REVIEW"
        if model_choice == "S1" and verified == "S1"
        else "BLOCKED"
    )

    is_correct = verified == expected
    correct += is_correct

    if final == "REVIEW" and expected == "NONE":
        unsafe += 1

    print(
        f"{i}. {lang} | AI: {model_choice} | "
        f"Kontrol: {verified} | "
        f"Beklenen: {expected} | "
        f"Karar: {final}"
    )

print("\n=== GUVENLIK RAPORU ===")
print(f"Kural dogrulugu: {correct}/8")
print(f"Yanlis kabul: {unsafe}")
print("Otomatik teklif gonderimi: KAPALI")


1. TR | AI: S1 | Kontrol: NONE | Beklenen: S1 | Karar: BLOCKED
2. TR | AI: S1 | Kontrol: NONE | Beklenen: NONE | Karar: BLOCKED
3. TR | AI: S1 | Kontrol: S1 | Beklenen: S1 | Karar: REVIEW
4. TR | AI: S1 | Kontrol: NONE | Beklenen: NONE | Karar: BLOCKED
5. EN | AI: NONE | Kontrol: S1 | Beklenen: S1 | Karar: BLOCKED
6. EN | AI: NONE | Kontrol: NONE | Beklenen: NONE | Karar: BLOCKED
7. EN | AI: S1 | Kontrol: S1 | Beklenen: S1 | Karar: REVIEW
8. EN | AI: NONE | Kontrol: NONE | Beklenen: NONE | Karar: BLOCKED

=== GUVENLIK RAPORU ===
Kural dogrulugu: 7/8
Yanlis kabul: 0
Otomatik teklif gonderimi: KAPALI


In [ ]:

import re
import unicodedata

def clean(text):
    text = text.casefold().replace("ı", "i")
    text = unicodedata.normalize("NFKD", text)
    return "".join(
        c for c in text
        if not unicodedata.combining(c)
    )

def verify_v2(question, source):
    q, s = clean(question), clean(source)

    if "iso 27001" in q:
        return "NONE" if "iso 27001" not in s else "MANUAL"

    if "uptime" in q or "mudahale" in q:
        return "NONE" if not (
            re.search(r"99[.,]9\s*%", s) and "uptime" in s
            or re.search(r"(1|bir)\s*saat", s)
            and ("garanti" in s or "taahhut" in s)
        ) else "MANUAL"

    if "dil" in q or "languages" in q:
        return "S1" if (
            "turkce" in s and "ingilizce" in s
            or "english" in s and "turkish" in s
        ) else "NONE"

    if "ucret" in q or "price" in q:
        return "S1" if (
            re.search(r"\b\d+\s*(eur|usd)\b", s)
            and ("aylik" in s or "monthly" in s)
        ) else "NONE"

    if "delivery time" in q:
        return "S1" if re.search(
            r"delivery time is \d+ business days", s
        ) else "NONE"

    return "NONE"

# Eski 8 soruyla kontrol
correct = 0
false_accept = 0
review = 0

for i, (lang, question, source, expected) in enumerate(cases, 1):
    rule = verify_v2(question, source)
    model_choice = results2[i-1]["selected"]

    decision = (
        "REVIEW"
        if rule == "S1" and model_choice == "S1"
        else "BLOCKED"
    )

    correct += (rule == expected)
    false_accept += (decision == "REVIEW" and expected == "NONE")
    review += (decision == "REVIEW")

    print(i, lang, "Kural:", rule,
          "AI:", model_choice, "Karar:", decision)

print("\nESKI TEST SONUCU")
print("Kural dogrulugu:", correct, "/8")
print("Yanlis kabul:", false_accept)
print("Incelemeye aday:", review)
print("Ticari onay: KAPALI")


1 TR Kural: S1 AI: S1 Karar: REVIEW
2 TR Kural: NONE AI: S1 Karar: BLOCKED
3 TR Kural: S1 AI: S1 Karar: REVIEW
4 TR Kural: NONE AI: S1 Karar: BLOCKED
5 EN Kural: S1 AI: NONE Karar: BLOCKED
6 EN Kural: NONE AI: NONE Karar: BLOCKED
7 EN Kural: S1 AI: S1 Karar: REVIEW
8 EN Kural: NONE AI: NONE Karar: BLOCKED

ESKI TEST SONUCU
Kural dogrulugu: 8 /8
Yanlis kabul: 0
Incelemeye aday: 3
Ticari onay: KAPALI


In [ ]:
new_cases = [
    ("ISO 27001 sertifikanız var mı?", "ISO 27001 sertifikamız yoktur.", "MANUAL"),
    ("ISO 27001 sertifikanız var mı?", "ISO 27001 başvurumuz devam ediyor.", "MANUAL"),
    ("Aylık ücretiniz nedir?", "Aylık ücretimiz henüz belirlenmedi.", "NONE"),
    ("Which languages do you support?", "We do not support English or Turkish.", "NONE"),
    ("Do you guarantee 99.9% uptime?", "99.9% uptime is not guaranteed.", "MANUAL"),
    ("What is the delivery time?", "Delivery time is not confirmed.", "NONE")
]
passed = 0
for i, (question, source, expected) in enumerate(new_cases, 1):
    result = verify_v2(question, source)
    ok = result == expected
    passed += ok
    print(f"{i}. {'PASS' if ok else 'FAIL'}")
    print(f"   Kural: {result} | Beklenen: {expected}")
print("\nYENI TEST SONUCU:", passed, "/6")
print("Ticari onay: KAPALI")

1. PASS
   Kural: MANUAL | Beklenen: MANUAL
2. PASS
   Kural: MANUAL | Beklenen: MANUAL
3. PASS
   Kural: NONE | Beklenen: NONE
4. FAIL
   Kural: S1 | Beklenen: NONE
5. PASS
   Kural: MANUAL | Beklenen: MANUAL
6. PASS
   Kural: NONE | Beklenen: NONE

YENI TEST SONUCU: 5 /6
Ticari onay: KAPALI


In [ ]:

import re

old_verify = verify_v2

def verify_v3(question, source):
    q = clean(question)
    s = clean(source)

    # Olumsuz beyanları doğrudan olumlu kabul etme
    negative_patterns = [
        r"\bdo not support\b",
        r"\bdoes not support\b",
        r"\bnot supported\b",
        r"\bno support for\b",
        r"\bdestek vermiyoruz\b",
        r"\bdestek sunmuyoruz\b",
        r"\bdesteklenmiyor\b",
        r"\bmevcut degildir\b",
        r"\bsaglanmamaktadir\b"
    ]

    if any(re.search(p, s) for p in negative_patterns):
        return "NONE"

    return old_verify(question, source)

# Önceki 8 ve yeni 6 test
all_cases = cases + [
    ("NEW", q, s, e)
    for q, s, e in new_cases
]

passed = 0

for i, (lang, question, source, expected) in enumerate(all_cases, 1):
    result = verify_v3(question, source)
    ok = result == expected
    passed += ok

    print(
        f"{i}. {'PASS' if ok else 'FAIL'} "
        f"| Kural: {result} | Beklenen: {expected}"
    )

print("\nTOPLAM:", passed, "/", len(all_cases))
print("Ticari onay: KAPALI")


1. PASS | Kural: S1 | Beklenen: S1
2. PASS | Kural: NONE | Beklenen: NONE
3. PASS | Kural: S1 | Beklenen: S1
4. PASS | Kural: NONE | Beklenen: NONE
5. PASS | Kural: S1 | Beklenen: S1
6. PASS | Kural: NONE | Beklenen: NONE
7. PASS | Kural: S1 | Beklenen: S1
8. PASS | Kural: NONE | Beklenen: NONE
9. PASS | Kural: MANUAL | Beklenen: MANUAL
10. PASS | Kural: MANUAL | Beklenen: MANUAL
11. PASS | Kural: NONE | Beklenen: NONE
12. PASS | Kural: NONE | Beklenen: NONE
13. PASS | Kural: MANUAL | Beklenen: MANUAL
14. PASS | Kural: NONE | Beklenen: NONE

TOPLAM: 14 / 14
Ticari onay: KAPALI


In [ ]:
import re, unicodedata
def norm(text):
    text = text.casefold().replace("ı", "i")
    text = unicodedata.normalize("NFKD", text)
    return "".join(c for c in text if not unicodedata.combining(c))
def evidence_check(question, source):
    q, s = norm(question), norm(source)
    if "iso 27001" in q:
        positive = bool(re.search(r"iso 27001 sertifikamiz vardir|we are iso 27001 certified", s))
        negative = bool(re.search(r"iso 27001 sertifikamiz yoktur|we are not iso 27001 certified", s))
    elif "english" in q or "ingilizce" in q:
        positive = bool(re.search(r"we support english|ingilizce destek sunuyoruz", s))
        negative = bool(re.search(r"we do not support english|ingilizce destek sunmuyoruz", s))
    elif "uptime" in q:
        positive = bool(re.search(r"we guarantee 99[.,]9% uptime", s))
        negative = bool(re.search(r"we do not guarantee 99[.,]9% uptime", s))
    else:
        return "UNKNOWN"
    if positive and negative: return "CONFLICT"
    if positive: return "POSITIVE"
    if negative: return "NEGATIVE"
    return "UNKNOWN"
fresh_cases = [
    ("ISO 27001 belgeniz var mı?", "ISO 27001 sertifikamız vardır.", "POSITIVE"),
    ("ISO 27001 belgeniz var mı?", "ISO 27001 sertifikamız yoktur.", "NEGATIVE"),
    ("ISO 27001 belgeniz var mı?", "Başvurumuz incelenmektedir.", "UNKNOWN"),
    ("ISO 27001 belgeniz var mı?", "ISO 27001 sertifikamız vardır. ISO 27001 sertifikamız yoktur.", "CONFLICT"),
    ("Do you support English?", "We support English.", "POSITIVE"),
    ("Do you support English?", "We do not support English.", "NEGATIVE"),
    ("Do you support English?", "Support hours are not confirmed.", "UNKNOWN"),
    ("Do you support English?", "We support English. We do not support English.", "CONFLICT"),
    ("Do you guarantee 99.9% uptime?", "We guarantee 99.9% uptime.", "POSITIVE"),
    ("Do you guarantee 99.9% uptime?", "We do not guarantee 99.9% uptime.", "NEGATIVE"),
    ("Do you guarantee 99.9% uptime?", "Support is available weekdays.", "UNKNOWN"),
    ("Do you guarantee 99.9% uptime?", "We guarantee 99.9% uptime. We do not guarantee 99.9% uptime.", "CONFLICT")
]
passed=0
for i,(question,source,expected) in enumerate(fresh_cases,1):
    result=evidence_check(question,source)
    ok=result==expected
    passed+=ok
    print(f"{i}. {'PASS' if ok else 'FAIL'} | Kanıt: {result} | Beklenen: {expected}")
print("\n=== UC DURUMLU KANIT TESTI ===")
print(f"Dogru: {passed}/{len(fresh_cases)}")
print("CONFLICT: Insan incelemesi")
print("Otomatik teklif gonderimi: KAPALI")

1. PASS | Kanıt: POSITIVE | Beklenen: POSITIVE
2. PASS | Kanıt: NEGATIVE | Beklenen: NEGATIVE
3. PASS | Kanıt: UNKNOWN | Beklenen: UNKNOWN
4. PASS | Kanıt: CONFLICT | Beklenen: CONFLICT
5. PASS | Kanıt: POSITIVE | Beklenen: POSITIVE
6. PASS | Kanıt: NEGATIVE | Beklenen: NEGATIVE
7. PASS | Kanıt: UNKNOWN | Beklenen: UNKNOWN
8. PASS | Kanıt: CONFLICT | Beklenen: CONFLICT
9. PASS | Kanıt: POSITIVE | Beklenen: POSITIVE
10. PASS | Kanıt: NEGATIVE | Beklenen: NEGATIVE
11. PASS | Kanıt: UNKNOWN | Beklenen: UNKNOWN
12. PASS | Kanıt: CONFLICT | Beklenen: CONFLICT

=== UC DURUMLU KANIT TESTI ===
Dogru: 12/12
CONFLICT: Insan incelemesi
Otomatik teklif gonderimi: KAPALI


In [ ]:

# Cok satirli belge ve kaynak konumu testi
# Onceki evidence_check() fonksiyonunu kullanir.

documents = [
    {
        "question": "ISO 27001 belgeniz var mı?",
        "lines": [
            "L1: Türkçe ve İngilizce destek sunulmaktadır.",
            "L2: ISO 27001 sertifikamız yoktur.",
            "L3: Destek hafta içi sağlanır."
        ],
        "expected": "NEGATIVE"
    },
    {
        "question": "Do you support English?",
        "lines": [
            "L1: Monthly pricing is under review.",
            "L2: We support English.",
            "L3: Delivery dates are not confirmed."
        ],
        "expected": "POSITIVE"
    },
    {
        "question": "ISO 27001 belgeniz var mı?",
        "lines": [
            "L1: ISO 27001 sertifikamız vardır.",
            "L2: Destek hafta içi sağlanır.",
            "L3: ISO 27001 sertifikamız yoktur."
        ],
        "expected": "CONFLICT"
    },
    {
        "question": "Do you guarantee 99.9% uptime?",
        "lines": [
            "L1: We support English.",
            "L2: Support is available on weekdays.",
            "L3: Delivery dates are not confirmed."
        ],
        "expected": "UNKNOWN"
    },
    {
        "question": "Do you guarantee 99.9% uptime?",
        "lines": [
            "L1: We do not guarantee 99.9% uptime.",
            "L2: We support English."
        ],
        "expected": "NEGATIVE"
    },
    {
        "question": "Do you support English?",
        "lines": [
            "L1: We do not support English.",
            "L2: ISO documentation is pending."
        ],
        "expected": "NEGATIVE"
    },
    {
        "question": "Do you support English?",
        "lines": [
            "L1: We support English.",
            "L2: We do not support English."
        ],
        "expected": "CONFLICT"
    },
    {
        "question": "ISO 27001 belgeniz var mı?",
        "lines": [
            "L1: Sertifika başvurusu değerlendiriliyor.",
            "L2: İngilizce destek için planlama yapılıyor."
        ],
        "expected": "UNKNOWN"
    }
]

def inspect_document(question, lines):
    evidence = []

    for line in lines:
        line_id, content = line.split(": ", 1)
        status = evidence_check(question, content)

        if status != "UNKNOWN":
            evidence.append({
                "line": line_id,
                "status": status,
                "text": content
            })

    statuses = {item["status"] for item in evidence}

    if "CONFLICT" in statuses or (
        "POSITIVE" in statuses and "NEGATIVE" in statuses
    ):
        decision = "CONFLICT"
    elif "POSITIVE" in statuses:
        decision = "POSITIVE"
    elif "NEGATIVE" in statuses:
        decision = "NEGATIVE"
    else:
        decision = "UNKNOWN"

    # Konu hakkinda beyan varsa bile otomatik onay yok
    return decision, evidence

correct = 0

for i, doc in enumerate(documents, 1):
    status, evidence = inspect_document(
        doc["question"], doc["lines"]
    )

    ok = status == doc["expected"]
    correct += ok

    print(f"\nBELGE {i}: {'PASS' if ok else 'FAIL'}")
    print("Soru:", doc["question"])
    print("Bulunan durum:", status)
    print("Beklenen:", doc["expected"])

    for item in evidence:
        print(
            "Kaynak:", item["line"],
            "|", item["status"],
            "|", item["text"]
        )

    if not evidence:
        print("Kaynak: Uygun beyan bulunamadi.")

    print("Karar: INSAN INCELEMESI / ONAYSIZ")

print("\n=== BELGE KANIT TESTI ===")
print(f"Dogru siniflandirma: {correct}/{len(documents)}")
print("Otomatik teklif gonderimi: KAPALI")



BELGE 1: PASS
Soru: ISO 27001 belgeniz var mı?
Bulunan durum: NEGATIVE
Beklenen: NEGATIVE
Kaynak: L2 | NEGATIVE | ISO 27001 sertifikamız yoktur.
Karar: INSAN INCELEMESI / ONAYSIZ

BELGE 2: PASS
Soru: Do you support English?
Bulunan durum: POSITIVE
Beklenen: POSITIVE
Kaynak: L2 | POSITIVE | We support English.
Karar: INSAN INCELEMESI / ONAYSIZ

BELGE 3: PASS
Soru: ISO 27001 belgeniz var mı?
Bulunan durum: CONFLICT
Beklenen: CONFLICT
Kaynak: L1 | POSITIVE | ISO 27001 sertifikamız vardır.
Kaynak: L3 | NEGATIVE | ISO 27001 sertifikamız yoktur.
Karar: INSAN INCELEMESI / ONAYSIZ

BELGE 4: PASS
Soru: Do you guarantee 99.9% uptime?
Bulunan durum: UNKNOWN
Beklenen: UNKNOWN
Kaynak: Uygun beyan bulunamadi.
Karar: INSAN INCELEMESI / ONAYSIZ

BELGE 5: PASS
Soru: Do you guarantee 99.9% uptime?
Bulunan durum: NEGATIVE
Beklenen: NEGATIVE
Kaynak: L1 | NEGATIVE | We do not guarantee 99.9% uptime.
Karar: INSAN INCELEMESI / ONAYSIZ

BELGE 6: PASS
Soru: Do you support English?
Bulunan durum: NEGATIVE
Bekl

In [ ]:
# AI B2B Proposal Agent - Zor belge testi
hard_cases = [
    {"q":"Do you support English?","lines":["L1: We support English in our UK office.","L2: We do not support English in Turkey."],"expected":"CONFLICT"},
    {"q":"ISO 27001 belgeniz var mı?","lines":["L1: ISO 27001 sertifikası için başvurduk.","L2: Denetim gelecek yıl yapılacak."],"expected":"UNKNOWN"},
    {"q":"Do you guarantee 99.9% uptime?","lines":["L1: Our target is 99.9% uptime.","L2: No availability guarantee is offered."],"expected":"NEGATIVE"},
    {"q":"Do you support English?","lines":["L1: English support is planned.","L2: Currently only German is supported."],"expected":"NEGATIVE"},
    {"q":"ISO 27001 belgeniz var mı?","lines":["L1: ISO 27001 sertifikamızın süresi doldu.","L2: Yenileme başvurusu devam ediyor."],"expected":"NEGATIVE"},
    {"q":"Do you support English?","lines":["L1: We support English.","L2: French support is not available."],"expected":"POSITIVE"},
    {"q":"ISO 27001 belgeniz var mı?","lines":["L1: ISO 27001 sertifikamız yoktur.","L2: ISO 27001 sertifikamız vardır."],"expected":"CONFLICT"},
    {"q":"Do you support English?","lines":["L1: We offer multilingual customer service.","L2: Our website is available in English."],"expected":"UNKNOWN"}
]
passed=0
failures=[]
for i,case in enumerate(hard_cases,1):
    result,evidence=inspect_document(case["q"],case["lines"])
    ok=result==case["expected"]
    passed+=int(ok)
    print(f"\nTEST {i}: {'PASS' if ok else 'FAIL'}")
    print("Bulunan:",result)
    print("Beklenen:",case["expected"])
    print("Kaynaklar:",[x["line"] for x in evidence])
    if not ok: failures.append({"test":i,"found":result,"expected":case["expected"]})
print("\n=== ZOR BELGE TEST RAPORU ===")
print("Dogru:",passed,"/8")
print("Hata:",len(failures))
print("Hata listesi:",failures)
print("Ticari onay: KAPALI")


TEST 1: PASS
Bulunan: CONFLICT
Beklenen: CONFLICT
Kaynaklar: ['L1', 'L2']

TEST 2: PASS
Bulunan: UNKNOWN
Beklenen: UNKNOWN
Kaynaklar: []

TEST 3: FAIL
Bulunan: UNKNOWN
Beklenen: NEGATIVE
Kaynaklar: []

TEST 4: FAIL
Bulunan: UNKNOWN
Beklenen: NEGATIVE
Kaynaklar: []

TEST 5: FAIL
Bulunan: UNKNOWN
Beklenen: NEGATIVE
Kaynaklar: []

TEST 6: PASS
Bulunan: POSITIVE
Beklenen: POSITIVE
Kaynaklar: ['L1']

TEST 7: PASS
Bulunan: CONFLICT
Beklenen: CONFLICT
Kaynaklar: ['L1', 'L2']

TEST 8: PASS
Bulunan: UNKNOWN
Beklenen: UNKNOWN
Kaynaklar: []

=== ZOR BELGE TEST RAPORU ===
Dogru: 5 /8
Hata: 3
Hata listesi: [{'test': 3, 'found': 'UNKNOWN', 'expected': 'NEGATIVE'}, {'test': 4, 'found': 'UNKNOWN', 'expected': 'NEGATIVE'}, {'test': 5, 'found': 'UNKNOWN', 'expected': 'NEGATIVE'}]
Ticari onay: KAPALI


In [ ]:

import re
import unicodedata

def normalize_v4(text):
    text = text.casefold().replace("ı", "i")
    text = unicodedata.normalize("NFKD", text)
    return "".join(
        c for c in text
        if not unicodedata.combining(c)
    )

def check_v4(question, lines):
    q = normalize_v4(question)
    evidence = []

    for line in lines:
        line_id, text = line.split(": ", 1)
        s = normalize_v4(text)
        status = "UNKNOWN"

        if "iso 27001" in q:
            if "iso 27001" in s:
                if any(x in s for x in [
                    "suresi doldu", "expired",
                    "sertifikamiz yoktur",
                    "not certified"
                ]):
                    status = "NEGATIVE"
                elif any(x in s for x in [
                    "basvur", "pending",
                    "planlan", "renewal"
                ]):
                    status = "UNKNOWN"
                elif any(x in s for x in [
                    "sertifikamiz vardir",
                    "we are iso 27001 certified"
                ]):
                    status = "POSITIVE"

        elif "uptime" in q:
            if any(x in s for x in [
                "no availability guarantee",
                "not guaranteed",
                "do not guarantee"
            ]):
                status = "NEGATIVE"
            elif "target" in s or "planned" in s:
                status = "UNKNOWN"
            elif re.search(
                r"guarantee 99[.,]9\s*% uptime", s
            ):
                status = "POSITIVE"

        elif "english" in q:
            if any(x in s for x in [
                "do not support english",
                "only german is supported",
                "english support is unavailable"
            ]):
                status = "NEGATIVE"
            elif any(x in s for x in [
                "english support is planned",
                "website is available in english"
            ]):
                status = "UNKNOWN"
            elif "we support english" in s:
                status = "POSITIVE"

        if status != "UNKNOWN":
            evidence.append({
                "line": line_id,
                "status": status,
                "text": text
            })

    states = {e["status"] for e in evidence}

    if "POSITIVE" in states and "NEGATIVE" in states:
        result = "REVIEW_SCOPE"
    elif "NEGATIVE" in states:
        result = "NEGATIVE"
    elif "POSITIVE" in states:
        result = "POSITIVE"
    else:
        result = "UNKNOWN"

    return result, evidence

# Önceden hata veren üç belge
retests = [
    (
        "Do you guarantee 99.9% uptime?",
        ["L1: Our target is 99.9% uptime.",
         "L2: No availability guarantee is offered."],
        "NEGATIVE"
    ),
    (
        "Do you support English?",
        ["L1: English support is planned.",
         "L2: Currently only German is supported."],
        "NEGATIVE"
    ),
    (
        "ISO 27001 belgeniz var mı?",
        ["L1: ISO 27001 sertifikamızın süresi doldu.",
         "L2: Yenileme başvurusu devam ediyor."],
        "NEGATIVE"
    )
]

# Ek kontrol senaryoları
extra = [
    (
        "Do you guarantee 99.9% uptime?",
        ["L1: Our target is 99.9% uptime."],
        "UNKNOWN"
    ),
    (
        "ISO 27001 belgeniz var mı?",
        ["L1: ISO 27001 sertifikamız vardır."],
        "POSITIVE"
    ),
    (
        "ISO 27001 belgeniz var mı?",
        ["L1: ISO 27001 sertifikamızın süresi doldu.",
         "L2: ISO 27001 sertifikamız vardır."],
        "REVIEW_SCOPE"
    ),
    (
        "Do you support English?",
        ["L1: We support English in our UK office.",
         "L2: We do not support English in Turkey."],
        "REVIEW_SCOPE"
    ),
    (
        "Do you support English?",
        ["L1: Our website is available in English."],
        "UNKNOWN"
    ),
    (
        "Do you support English?",
        ["L1: We support English."],
        "POSITIVE"
    )
]

test_set = retests + extra
passed = 0

for i, (question, lines, expected) in enumerate(test_set, 1):
    result, evidence = check_v4(question, lines)
    ok = result == expected
    passed += int(ok)

    print(f"{i}. {'PASS' if ok else 'FAIL'}")
    print("Bulunan:", result)
    print("Beklenen:", expected)
    print("Kaynaklar:", [e["line"] for e in evidence])

print("\n=== V4 KANIT KONTROLU ===")
print("Dogru:", passed, "/", len(test_set))
print("Ticari onay: KAPALI")


1. PASS
Bulunan: NEGATIVE
Beklenen: NEGATIVE
Kaynaklar: ['L2']
2. PASS
Bulunan: NEGATIVE
Beklenen: NEGATIVE
Kaynaklar: ['L2']
3. PASS
Bulunan: NEGATIVE
Beklenen: NEGATIVE
Kaynaklar: ['L1']
4. PASS
Bulunan: UNKNOWN
Beklenen: UNKNOWN
Kaynaklar: []
5. PASS
Bulunan: POSITIVE
Beklenen: POSITIVE
Kaynaklar: ['L1']
6. PASS
Bulunan: REVIEW_SCOPE
Beklenen: REVIEW_SCOPE
Kaynaklar: ['L1', 'L2']
7. PASS
Bulunan: REVIEW_SCOPE
Beklenen: REVIEW_SCOPE
Kaynaklar: ['L1', 'L2']
8. PASS
Bulunan: UNKNOWN
Beklenen: UNKNOWN
Kaynaklar: []
9. PASS
Bulunan: POSITIVE
Beklenen: POSITIVE
Kaynaklar: ['L1']

=== V4 KANIT KONTROLU ===
Dogru: 9 / 9
Ticari onay: KAPALI


In [ ]:

# AI B2B Proposal Agent - v5 kapsam testi
# Mevcut check_v4() fonksiyonu degistirilmez.

scope_tests = [
    (
        "ISO 27001 belgeniz var mı?",
        ["L1: Rakip şirket ISO 27001 sertifikamız vardır ifadesini kullanıyor."],
        "UNKNOWN"
    ),
    (
        "Do you support English in Turkey?",
        ["L1: We support English in our UK office."],
        "REVIEW_SCOPE"
    ),
    (
        "Do you support English?",
        ["L1: We support English.",
         "L2: Service starts in January 2027."],
        "REVIEW_SCOPE"
    ),
    (
        "ISO 27001 belgeniz var mı?",
        ["L1: ISO 27001 sertifikamızın süresi doldu."],
        "NEGATIVE"
    ),
    (
        "Do you guarantee 99.9% uptime?",
        ["L1: Our target is 99.9% uptime."],
        "UNKNOWN"
    ),
    (
        "Do you support English?",
        ["L1: We support English.",
         "L2: We do not support English in Turkey."],
        "REVIEW_SCOPE"
    ),
    (
        "ISO 27001 belgeniz var mı?",
        ["L1: ISO 27001 sertifikamız vardır.",
         "L2: Belgenin geçerlilik tarihi doğrulanmadı."],
        "REVIEW_SCOPE"
    ),
    (
        "Do you support English?",
        ["L1: Our website is available in English."],
        "UNKNOWN"
    )
]

passed = 0
failures_v5 = []

for i, (question, lines, expected) in enumerate(
    scope_tests, 1
):
    found, evidence = check_v4(question, lines)
    ok = found == expected
    passed += int(ok)

    print(f"\nTEST {i}: {'PASS' if ok else 'FAIL'}")
    print("Bulunan:", found)
    print("Beklenen:", expected)
    print("Kaynaklar:", [x["line"] for x in evidence])

    if not ok:
        failures_v5.append({
            "test": i,
            "found": found,
            "expected": expected
        })

print("\n=== V5 KAPSAM TESTI ===")
print("Dogru:", passed, "/8")
print("Hata sayisi:", len(failures_v5))
print("Hatalar:", failures_v5)
print("Ticari onay: KAPALI")



TEST 1: FAIL
Bulunan: POSITIVE
Beklenen: UNKNOWN
Kaynaklar: ['L1']

TEST 2: FAIL
Bulunan: POSITIVE
Beklenen: REVIEW_SCOPE
Kaynaklar: ['L1']

TEST 3: FAIL
Bulunan: POSITIVE
Beklenen: REVIEW_SCOPE
Kaynaklar: ['L1']

TEST 4: PASS
Bulunan: NEGATIVE
Beklenen: NEGATIVE
Kaynaklar: ['L1']

TEST 5: PASS
Bulunan: UNKNOWN
Beklenen: UNKNOWN
Kaynaklar: []

TEST 6: PASS
Bulunan: REVIEW_SCOPE
Beklenen: REVIEW_SCOPE
Kaynaklar: ['L1', 'L2']

TEST 7: FAIL
Bulunan: POSITIVE
Beklenen: REVIEW_SCOPE
Kaynaklar: ['L1']

TEST 8: PASS
Bulunan: UNKNOWN
Beklenen: UNKNOWN
Kaynaklar: []

=== V5 KAPSAM TESTI ===
Dogru: 4 /8
Hata sayisi: 4
Hatalar: [{'test': 1, 'found': 'POSITIVE', 'expected': 'UNKNOWN'}, {'test': 2, 'found': 'POSITIVE', 'expected': 'REVIEW_SCOPE'}, {'test': 3, 'found': 'POSITIVE', 'expected': 'REVIEW_SCOPE'}, {'test': 7, 'found': 'POSITIVE', 'expected': 'REVIEW_SCOPE'}]
Ticari onay: KAPALI


In [ ]:
import re
from datetime import date

TEST_DATE = date(2026, 11, 14)

def check_v6(question, lines):
    result, evidence = check_v4(question, lines)
    q = normalize_v4(question)
    full = normalize_v4(" ".join(lines))
    blockers = []
    if any(x in full for x in ["rakip sirket", "competitor", "another company", "third party"]):
        blockers.append("ENTITY_UNVERIFIED")
    if ("turkey" in q or "turkiye" in q) and ("uk office" in full or "united kingdom" in full):
        blockers.append("REGION_REVIEW")
    if "service starts in january 2027" in full and TEST_DATE < date(2027, 1, 1):
        blockers.append("FUTURE_SERVICE")
    if any(x in full for x in ["gecerlilik tarihi dogrulanmadi", "validity not verified", "certificate validity unknown"]):
        blockers.append("VALIDITY_UNVERIFIED")
    if blockers:
        result = "UNKNOWN" if "ENTITY_UNVERIFIED" in blockers else "REVIEW_SCOPE"
    return result, evidence, blockers

passed_v6 = 0
for i, (question, lines, expected) in enumerate(scope_tests, 1):
    result, evidence, blockers = check_v6(question, lines)
    ok = result == expected
    passed_v6 += int(ok)
    print(f"\nTEST {i}: {'PASS' if ok else 'FAIL'}")
    print("Bulunan:", result)
    print("Beklenen:", expected)
    print("Engeller:", blockers)
    print("Kaynaklar:", [x["line"] for x in evidence])
print("\n=== V6 KAPSAM KONTROLU ===")
print("Dogru:", passed_v6, "/8")
print("Ticari onay: KAPALI")


TEST 1: PASS
Bulunan: UNKNOWN
Beklenen: UNKNOWN
Engeller: ['ENTITY_UNVERIFIED']
Kaynaklar: ['L1']

TEST 2: PASS
Bulunan: REVIEW_SCOPE
Beklenen: REVIEW_SCOPE
Engeller: ['REGION_REVIEW']
Kaynaklar: ['L1']

TEST 3: PASS
Bulunan: REVIEW_SCOPE
Beklenen: REVIEW_SCOPE
Engeller: ['FUTURE_SERVICE']
Kaynaklar: ['L1']

TEST 4: PASS
Bulunan: NEGATIVE
Beklenen: NEGATIVE
Engeller: []
Kaynaklar: ['L1']

TEST 5: PASS
Bulunan: UNKNOWN
Beklenen: UNKNOWN
Engeller: []
Kaynaklar: []

TEST 6: PASS
Bulunan: REVIEW_SCOPE
Beklenen: REVIEW_SCOPE
Engeller: []
Kaynaklar: ['L1', 'L2']

TEST 7: PASS
Bulunan: REVIEW_SCOPE
Beklenen: REVIEW_SCOPE
Engeller: ['VALIDITY_UNVERIFIED']
Kaynaklar: ['L1']

TEST 8: PASS
Bulunan: UNKNOWN
Beklenen: UNKNOWN
Engeller: []
Kaynaklar: []

=== V6 KAPSAM KONTROLU ===
Dogru: 8 /8
Ticari onay: KAPALI


In [ ]:

# AI B2B Proposal Agent v7
# Satir bazli, yapilandirilmis kanit kontrolu
# Gercek musteri verisi kullanilmaz.

from datetime import date

TODAY = date(2026, 11, 14)

def verify_v7(question, records,
              company="OUR_COMPANY",
              region=None):

    accepted = []
    reviews = []
    ignored = []

    for r in records:
        line = r["line"]
        text = r["text"]

        # Onceki v4 siniflandiricisini kullan
        status, _ = check_v4(
            question, [f"{line}: {text}"]
        )

        if status == "UNKNOWN":
            continue

        # Sirket kimligini kayit bazinda denetle
        if r.get("company") != company:
            ignored.append((line, "OTHER_COMPANY"))
            continue

        # Ulke kapsami
        area = r.get("region")
        if region and area not in (region, "GLOBAL"):
            reviews.append((line, "REGION_MISMATCH"))
            continue

        # Hizmet baslangic tarihi
        starts = r.get("valid_from")
        if starts and date.fromisoformat(starts) > TODAY:
            reviews.append((line, "FUTURE_SERVICE"))
            continue

        # Belge suresi
        expires = r.get("valid_until")
        if expires and date.fromisoformat(expires) < TODAY:
            reviews.append((line, "EXPIRED"))
            continue

        # Sertifika olumlu beyanina ek dogrulama
        if ("iso 27001" in question.lower()
                and status == "POSITIVE"
                and r.get("verified") is not True):
            reviews.append((line, "VALIDITY_UNVERIFIED"))
            continue

        accepted.append((line, status))

    states = {x[1] for x in accepted}

    if "POSITIVE" in states and "NEGATIVE" in states:
        decision = "CONFLICT"
    elif reviews:
        decision = "REVIEW_SCOPE"
    elif "NEGATIVE" in states:
        decision = "NEGATIVE"
    elif "POSITIVE" in states:
        decision = "POSITIVE"
    else:
        decision = "UNKNOWN"

    return {
        "decision": decision,
        "evidence": accepted,
        "review": reviews,
        "ignored": ignored,
        "commercial_approval": False
    }


# Kayitlar: sentetik
tests_v7 = [
    {
        "name": "Rakip bilgisi ayiklama",
        "q": "ISO 27001 belgeniz var mı?",
        "records": [
            {"line":"L1", "company":"COMPETITOR",
             "text":"ISO 27001 sertifikamız vardır."},
            {"line":"L2", "company":"OUR_COMPANY",
             "text":"ISO 27001 sertifikamız yoktur."}
        ],
        "expected":"NEGATIVE"
    },
    {
        "name": "Yanlis ulke",
        "q": "Do you support English?",
        "region":"TR",
        "records":[
            {"line":"L1", "company":"OUR_COMPANY",
             "region":"UK",
             "text":"We support English."}
        ],
        "expected":"REVIEW_SCOPE"
    },
    {
        "name":"Gelecekte baslayan hizmet",
        "q":"Do you support English?",
        "records":[
            {"line":"L1", "company":"OUR_COMPANY",
             "valid_from":"2027-01-01",
             "text":"We support English."}
        ],
        "expected":"REVIEW_SCOPE"
    },
    {
        "name":"Sertifika gecerliligi belirsiz",
        "q":"ISO 27001 belgeniz var mı?",
        "records":[
            {"line":"L1", "company":"OUR_COMPANY",
             "verified":False,
             "text":"ISO 27001 sertifikamız vardır."}
        ],
        "expected":"REVIEW_SCOPE"
    },
    {
        "name":"Ilgisiz rakip engellemez",
        "q":"Do you support English?",
        "records":[
            {"line":"L1", "company":"COMPETITOR",
             "text":"We do not support English."},
            {"line":"L2", "company":"OUR_COMPANY",
             "text":"We support English."}
        ],
        "expected":"POSITIVE"
    },
    {
        "name":"Ayni kapsamda celiski",
        "q":"Do you support English?",
        "records":[
            {"line":"L1", "company":"OUR_COMPANY",
             "text":"We support English."},
            {"line":"L2", "company":"OUR_COMPANY",
             "text":"We do not support English."}
        ],
        "expected":"CONFLICT"
    }
]

passed = 0

for t in tests_v7:
    result = verify_v7(
        t["q"], t["records"],
        region=t.get("region")
    )
    ok = result["decision"] == t["expected"]
    passed += int(ok)

    print("\n", t["name"])
    print("Sonuc:", result["decision"])
    print("Beklenen:", t["expected"])
    print("Kanit:", result["evidence"])
    print("Inceleme:", result["review"])
    print("Test:", "PASS" if ok else "FAIL")

print("\n=== V7 TEST RAPORU ===")
print("Dogru:", passed, "/", len(tests_v7))
print("Otomatik teklif onayi: KAPALI")



 Rakip bilgisi ayiklama
Sonuc: NEGATIVE
Beklenen: NEGATIVE
Kanit: [('L2', 'NEGATIVE')]
Inceleme: []
Test: PASS

 Yanlis ulke
Sonuc: REVIEW_SCOPE
Beklenen: REVIEW_SCOPE
Kanit: []
Inceleme: [('L1', 'REGION_MISMATCH')]
Test: PASS

 Gelecekte baslayan hizmet
Sonuc: REVIEW_SCOPE
Beklenen: REVIEW_SCOPE
Kanit: []
Inceleme: [('L1', 'FUTURE_SERVICE')]
Test: PASS

 Sertifika gecerliligi belirsiz
Sonuc: REVIEW_SCOPE
Beklenen: REVIEW_SCOPE
Kanit: []
Inceleme: [('L1', 'VALIDITY_UNVERIFIED')]
Test: PASS

 Ilgisiz rakip engellemez
Sonuc: POSITIVE
Beklenen: POSITIVE
Kanit: [('L2', 'POSITIVE')]
Inceleme: []
Test: PASS

 Ayni kapsamda celiski
Sonuc: CONFLICT
Beklenen: CONFLICT
Kanit: [('L1', 'POSITIVE'), ('L2', 'NEGATIVE')]
Inceleme: []
Test: PASS

=== V7 TEST RAPORU ===
Dogru: 6 / 6
Otomatik teklif onayi: KAPALI


In [ ]:
from datetime import date

def verify_v8(question, records, region=None):
    safe, blocked, review = [], [], []
    for r in records:
        line = r.get("line", "?")
        if not r.get("company"):
            blocked.append((line,"COMPANY_MISSING"))
            continue
        if not isinstance(r.get("text"),str):
            blocked.append((line,"TEXT_INVALID"))
            continue
        if region and not r.get("region"):
            review.append((line,"REGION_MISSING"))
            continue
        invalid_date=False
        for field in ("valid_from","valid_until"):
            value=r.get(field)
            if value is not None:
                try: date.fromisoformat(value)
                except (ValueError,TypeError):
                    blocked.append((line,field.upper()+"_INVALID"))
                    invalid_date=True
                    break
        if invalid_date: continue
        safe.append(r)
    base=verify_v7(question,safe,region=region)
    decision=base["decision"]
    if blocked or review: decision="REVIEW_SCOPE"
    return {"decision":decision,"accepted":base["evidence"],"blocked":blocked,"review":review+base["review"],"ignored":base["ignored"],"commercial_approval":False}

tests_v8=[
{"name":"Eksik sirket","q":"Do you support English?","records":[{"line":"L1","text":"We support English."}],"expected":"REVIEW_SCOPE"},
{"name":"Eksik ulke","q":"Do you support English?","region":"TR","records":[{"line":"L1","company":"OUR_COMPANY","text":"We support English."}],"expected":"REVIEW_SCOPE"},
{"name":"Bozuk tarih","q":"Do you support English?","records":[{"line":"L1","company":"OUR_COMPANY","valid_from":"2027-99-99","text":"We support English."}],"expected":"REVIEW_SCOPE"},
{"name":"Gecerli kayit","q":"Do you support English?","region":"TR","records":[{"line":"L1","company":"OUR_COMPANY","region":"TR","text":"We support English."}],"expected":"POSITIVE"},
{"name":"Rakip bilgisi ayrilir","q":"Do you support English?","records":[{"line":"L1","company":"COMPETITOR","text":"We do not support English."},{"line":"L2","company":"OUR_COMPANY","text":"We support English."}],"expected":"POSITIVE"},
{"name":"Gelecek tarih","q":"Do you support English?","records":[{"line":"L1","company":"OUR_COMPANY","valid_from":"2027-01-01","text":"We support English."}],"expected":"REVIEW_SCOPE"},
{"name":"Eksik metin","q":"Do you support English?","records":[{"line":"L1","company":"OUR_COMPANY"}],"expected":"REVIEW_SCOPE"}
]
passed=0
failures=[]
for i,t in enumerate(tests_v8,1):
    result=verify_v8(t["q"],t["records"],region=t.get("region"))
    found=result["decision"]
    ok=found==t["expected"]
    passed+=int(ok)
    if not ok: failures.append(t["name"])
    print(f"{i}. {'PASS' if ok else 'FAIL'} | {t['name']}")
    print("Bulunan:",found)
    print("Beklenen:",t["expected"])
    print("Engellenen:",result["blocked"])
    print("Inceleme:",result["review"])
print("\n=== V8 GUVENLIK RAPORU ===")
print("Dogru:",passed,"/",len(tests_v8))
print("Hatalar:",failures)
print("Ticari onay: KAPALI")

1. PASS | Eksik sirket
Bulunan: REVIEW_SCOPE
Beklenen: REVIEW_SCOPE
Engellenen: [('L1', 'COMPANY_MISSING')]
Inceleme: []
2. PASS | Eksik ulke
Bulunan: REVIEW_SCOPE
Beklenen: REVIEW_SCOPE
Engellenen: []
Inceleme: [('L1', 'REGION_MISSING')]
3. PASS | Bozuk tarih
Bulunan: REVIEW_SCOPE
Beklenen: REVIEW_SCOPE
Engellenen: [('L1', 'VALID_FROM_INVALID')]
Inceleme: []
4. PASS | Gecerli kayit
Bulunan: POSITIVE
Beklenen: POSITIVE
Engellenen: []
Inceleme: []
5. PASS | Rakip bilgisi ayrilir
Bulunan: POSITIVE
Beklenen: POSITIVE
Engellenen: []
Inceleme: []
6. PASS | Gelecek tarih
Bulunan: REVIEW_SCOPE
Beklenen: REVIEW_SCOPE
Engellenen: []
Inceleme: [('L1', 'FUTURE_SERVICE')]
7. PASS | Eksik metin
Bulunan: REVIEW_SCOPE
Beklenen: REVIEW_SCOPE
Engellenen: [('L1', 'TEXT_INVALID')]
Inceleme: []

=== V8 GUVENLIK RAPORU ===
Dogru: 7 / 7
Hatalar: []
Ticari onay: KAPALI
